# Analytics Store Tutorial

<div class="admonition tip"><p class="admonition-title">What you'll learn</p>

- How to share one `AnalyticsStore` across multiple capabilities
- What each capability's record looks like in the store
- How to query a single capability's results with SQL
- How to JOIN across capabilities on a shared `dataset_id`
- How to read the results from outside Python

</div>

## What is the Analytics Store?

The Analytics Store is a thin write-and-query layer over your capability outputs. Each capability emits one or more records via `.extract()`; the store writes them to a backing format (Parquet by default) and lets you query the result with SQL. Records from different capabilities live in different tables but share key columns — most importantly `dataset_id` — so you can answer questions that span capabilities without re-running anything.

```mermaid
flowchart LR
    A[COCO dataset] --> B1[Dataeval Bias]
    A --> B2[Dataeval Feasibility]
    A --> B3[Dataeval Shift]
    A --> B4[NRTK Robustness]
    A --> B5[XAITK Explainable]
    B1 --> S[(Analytics Store)]
    B2 --> S
    B3 --> S
    B4 --> S
    B5 --> S
    S --> Q[Cross-capability SQL JOINs]
```


<div class="admonition note"><p class="admonition-title">Scope</p>

`Dataeval Bias`, `Dataeval Feasibility`, and `Dataeval Shift`     are executed for real against the bundled COCO sample.     `NRTK Robustness` and `XAITK Explainable` would take too long to     run in a docs build, so this tutorial constructs simulated     records to demonstrate the store contract — see the per-tool     tutorials for end-to-end real runs.

</div>

## Setup

We create a single `AnalyticsStore` backed by a temporary directory. Every capability in this tutorial writes into this same store, which is what makes the cross-capability JOINs at the end work.

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

REPO_ROOT = Path.cwd().parents[1]
COCO_ROOT = REPO_ROOT / "tests/data_for_tests/coco_dataset"
COCO_RESIZED_ROOT = REPO_ROOT / "tests/data_for_tests/coco_resized_val2017"

if not (COCO_ROOT / "ann_file.json").exists():
    raise FileNotFoundError(f"missing {COCO_ROOT}")
if not (COCO_RESIZED_ROOT / "instances_val2017_resized_6.json").exists():
    raise FileNotFoundError(f"missing {COCO_RESIZED_ROOT}")

store_dir = tempfile.mkdtemp(prefix="analytics_store_tutorial_")
store = AnalyticsStore(ParquetBackend(store_dir))
print("store backed by:", store_dir)

## Load the dataset

All five capabilities run against the same `coco-example` dataset. `DataevalShift` additionally needs a second dataset to compare against — we use the resized COCO bundle as the evaluation dataset. Passing an explicit `dataset_id=` keeps the IDs stable across capabilities, which is what makes the JOINs at the end of this tutorial work.

In [ ]:
dataset = CocoDetectionDataset(
    root=str(COCO_ROOT),
    ann_file=str(COCO_ROOT / "ann_file.json"),
    dataset_id="coco-example",
)
shift_eval_dataset = CocoDetectionDataset(
    root=str(COCO_RESIZED_ROOT),
    ann_file=str(COCO_RESIZED_ROOT / "instances_val2017_resized_6.json"),
    dataset_id="coco-resized",
)
print(f"primary  : {dataset.metadata['id']} ({len(dataset)} images)")
print(f"shift_eval: {shift_eval_dataset.metadata['id']} ({len(shift_eval_dataset)} images)")

## Capability 1 — Dataeval Bias

`DataevalBiasBase` reports dataset-level coverage, balance, and diversity statistics. See [the bias tutorial](dataeval_bias_tutorial.html) for the full walkthrough — this section just shows how its results land in the store.

In [ ]:
from checkmaite.core._common.dataeval_bias_capability import DataevalBiasBase

bias_run = DataevalBiasBase().run(datasets=[dataset], use_cache=False)
bias_records = bias_run.extract()
print(f"extracted {len(bias_records)} bias record(s)")
bias_records[0]

In [ ]:
store.write([bias_run])
store.list_tables()

In [ ]:
store.query_sql("SELECT dataset_id, coverage_uncovered_ratio, balance_mean, diversity_mean " "FROM dataeval_bias")

## Capability 2 — Dataeval Feasibility

`DataevalFeasibility` reports Bayes-error-rate (BER) bounds and object-detection health stats — small-object ratio, truncated-bbox ratio, and any health warnings. See [the feasibility tutorial](dataeval_feasibility_tutorial.html) for the full treatment.

In [ ]:
from checkmaite.core.object_detection import DataevalFeasibility

feasibility_run = DataevalFeasibility().run(datasets=[dataset], use_cache=False)
feasibility_records = feasibility_run.extract()
print(f"extracted {len(feasibility_records)} feasibility record(s)")
feasibility_records[0]

In [ ]:
store.write([feasibility_run])
store.query_sql(
    "SELECT dataset_id, ber_upper, ber_lower, small_object_ratio, " "health_warning_count FROM dataeval_feasibility"
)

## Capability 3 — Dataeval Shift

<div class="admonition info"><p class="admonition-title">Shift compares two datasets</p>

Unlike the single-dataset capabilities above, `DataevalShift` takes a     reference dataset and an evaluation dataset and reports drift     (MMD, CVM, KS) and out-of-distribution statistics. Its record stores     **two** dataset IDs — `reference_dataset_id` and `evaluation_dataset_id` —     which matters when you start writing cross-capability JOINs.

</div>

See [the shift tutorial](dataeval_shift_tutorial.html) for the full walkthrough.

In [ ]:
from checkmaite.core.object_detection import DataevalShift

shift_run = DataevalShift().run(
    datasets=[dataset, shift_eval_dataset],
    use_cache=False,
)
shift_records = shift_run.extract()
print(f"extracted {len(shift_records)} shift record(s)")
shift_records[0]

In [ ]:
store.write([shift_run])
store.query_sql(
    "SELECT reference_dataset_id, evaluation_dataset_id, " "mmd_drifted, mmd_distance, ood_ratio " "FROM dataeval_shift"
)

## Capability 4 — NRTK Robustness (simulated)

<div class="admonition info"><p class="admonition-title">These records are hand-built for the tutorial</p>

Running NRTK against real perturbers is slow and out of scope here.     We construct a small set of `NrtkRobustnessRecord` objects directly     so the JOIN finale below has rows to work with. For an end-to-end     NRTK walkthrough, see [the nrtk tutorial](nrtk_tutorial.html).

</div>
The records simulate a 5-point brightness sweep on the `coco-resized` dataset — the same dataset we used as the evaluation pair for shift, so we can JOIN drift findings against robustness findings on equal footing.

In [ ]:
from checkmaite.core._common.nrtk_robustness_capability import NrtkRobustnessRecord

thetas = [0.6, 0.8, 1.0, 1.2, 1.4]
accuracies = [0.62, 0.74, 0.81, 0.69, 0.55]

nrtk_records = [
    NrtkRobustnessRecord(
        run_uid="tutorial-nrtk-run",
        dataset_id="coco-resized",
        model_id="resnet50",
        metric_id="coco_metrics",
        perturber_class="BrightnessPerturber",
        perturber_type="Brightness Perturber",
        theta_key="factor",
        theta_index=i,
        theta_value=theta,
        metric_key="accuracy",
        metric_value=acc,
        is_primary=True,
    )
    for i, (theta, acc) in enumerate(zip(thetas, accuracies, strict=False))
]
len(nrtk_records)

In [ ]:
# nrtk records are written via the backend directly since they are not
# produced by a capability .run() in this tutorial.
store._backend.write(nrtk_records)  # noqa: SLF001 (intentional for the demo)
store.query_sql("SELECT theta_value, metric_value FROM nrtk_robustness " "WHERE is_primary = true ORDER BY theta_index")

## Capability 5 — XAITK Explainable (simulated)

<div class="admonition info"><p class="admonition-title">These records are hand-built for the tutorial</p>

XAITK produces per-image (or per-detection) saliency summaries — a     different schema shape than the per-dataset capabilities above. We     construct a few records directly to show the table layout. For the     real walkthrough see [the xaitk tutorial](xaitk_tutorial.html).

</div>

In [ ]:
from checkmaite.core._common.xaitk_explainable_capability import (
    XaitkExplainableRecord,
)

xaitk_records = [
    XaitkExplainableRecord(
        run_uid="tutorial-xaitk-run",
        dataset_id="coco-example",
        model_id="resnet50",
        saliency_generator_type="RISEStack",
        image_index=i,
        gt_label=label,
        mean_saliency=mean,
        max_saliency=mx,
        std_saliency=std,
        positive_saliency_ratio=pos,
    )
    for i, (label, mean, mx, std, pos) in enumerate(
        [
            ("cat", 0.34, 0.88, 0.18, 0.62),
            ("dog", 0.41, 0.93, 0.21, 0.71),
            ("car", 0.28, 0.80, 0.15, 0.55),
        ]
    )
]
len(xaitk_records)

In [ ]:
store._backend.write(xaitk_records)  # noqa: SLF001
store.query_sql(
    "SELECT image_index, gt_label, mean_saliency, max_saliency " "FROM xaitk_explainable ORDER BY image_index"
)

## Cross-capability JOINs

Every capability above wrote into the same store, keyed by `dataset_id`. That single design choice is what makes the next three queries possible — we can ask questions that span capabilities without leaving SQL.

First, confirm what landed in the store:

In [ ]:
sorted(store.list_tables())

### JOIN 1 — Everything dataeval knows about `coco-example`

Bias and feasibility both key by `dataset_id`. Joining them in one query gives a multi-capability profile of the dataset in a single row.

In [ ]:
store.query_sql(
    "SELECT "
    "    b.dataset_id, "
    "    b.coverage_uncovered_ratio, "
    "    b.balance_mean, "
    "    b.diversity_mean, "
    "    f.ber_upper, "
    "    f.ber_lower, "
    "    f.small_object_ratio, "
    "    f.health_warning_count "
    "FROM dataeval_bias b "
    "JOIN dataeval_feasibility f USING (dataset_id)"
)

### JOIN 2 — Drift vs robustness on the same data

`dataeval_shift` doesn't have a single `dataset_id` — it has `reference_dataset_id` and `evaluation_dataset_id`. To JOIN against the single-dataset capabilities, we match on the side we care about. Here we put each dataset's drift verdict side-by-side with the model's robustness curve on the same data, so a reviewer can read drift status and robustness in one query.

In [ ]:
store.query_sql(
    "SELECT "
    "    s.evaluation_dataset_id AS dataset_id, "
    "    s.mmd_drifted, "
    "    s.mmd_distance, "
    "    s.ood_ratio, "
    "    n.perturber_type, "
    "    n.theta_value, "
    "    n.metric_value "
    "FROM dataeval_shift s "
    "JOIN nrtk_robustness n "
    "  ON s.evaluation_dataset_id = n.dataset_id "
    "WHERE n.is_primary = true AND n.metric_key = 'accuracy' "
    "ORDER BY n.theta_index"
)

### JOIN 3 — Provenance via `runs`

Every `store.write()` also populates the `runs` table with one row per (run, capability, entity). That table is your audit trail — joinable back to any capability table by `run_uid`.

In [ ]:
store.query_sql(
    "SELECT capability_table, entity_type, entity_id, run_uid, created_at "
    "FROM runs ORDER BY capability_table, created_at"
)

<div class="admonition tip"><p class="admonition-title">Why this matters</p>

Each capability's tutorial shows you *how* to run it. The analytics     store is what lets you put those results in one place and ask     cross-capability questions later — without re-running anything.

</div>

## Using results outside Python

The Parquet backend writes plain `.parquet` files — one or more per table. You can point any Parquet-aware tool (DuckDB CLI, pandas, polars from a different process, Spark, …) at the directory and query it without checkmaite installed.

In [ ]:
for path in sorted(Path(store_dir).rglob("*.parquet")):
    print(path.relative_to(store_dir))

## Distributed job submission

This tutorial writes results locally with `store.write([run])`. Under distributed job submission the responsibilities split:

- the **client** chooses the durable store location,
- the **worker** needs that information to persist results,
- and the **client** later reads from that same location.

So distributed runs require explicit configuration:

```python
from checkmaite.jobs import configure_job_backend

configure_job_backend(
    "ray",
    analytics_store={"backend": "parquet", "uri": "./analytics_store"},
)
```

`checkmaite.jobs._store` defines `AnalyticsStoreConfig`; the job backend forwards this config on submissions so workers write to the intended store. See [Analytics store in distributed execution](../development/job_submission/analytics_store.html) and [Job backend configuration](../development/job_submission/configure_job_backend.html).